In [1]:
from docling.document_converter import DocumentConverter
import fitz  # PyMuPDF

C:\Users\henry\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def redact_hyperlinked_text(input_path, output_path):
    doc = fitz.open(input_path)
    
    for page in doc:
        # get_links() returns a list of dictionaries representing all clickable areas
        links = page.get_links()
        
        for link in links:
            # link["from"] contains the exact bounding box (fitz.Rect) of the hyperlink
            link_rect = link["from"]
            
            # Add a redaction annotation over this specific box, filled with white
            page.add_redact_annot(link_rect, fill=(1, 1, 1))
        
        # Apply the redactions, physically wiping the text under the boxes
        page.apply_redactions()
        
    doc.save(output_path)
    doc.close()
    print("All hyperlinked elements redacted. Clean PDF saved.")

In [6]:
def get_tables(doc_str: str) -> list[str]:
    blocks = doc_str.split('\n\n')
    table_strings = []

    for i, block in enumerate(blocks):
        # A standard Markdown table contains a header separator row like "|---"
        if '|---' in block or '| ---' in block:
            # Add the table itself
            table_strings.append(f"**[Table Data]**\n{block.strip()}")
            table_strings.append("\n") # Add spacing between different table extractions

    # 3. Join the extracted blocks back into a single string
    return table_strings


def get_surrounding_paragraphs(doc_str: str, num_context_paragraphs: int = 1) -> list[tuple[str, str]]:
    blocks = doc_str.split('\n\n')
    surrounding_paragraphs = []

    for i, block in enumerate(blocks):
        # A standard Markdown table contains a header separator row like "|---"
        if '|---' in block or '| ---' in block:
            preceding_str = ""
            succeeding_str = ""
            
            # Grab paragraphs before the table (often the caption or introduction)
            preceding_paragraphs = []
            for j in range(1, num_context_paragraphs + 1):
                if i - j >= 0:
                    prev_block = blocks[i - j].strip()
                    if prev_block:
                        preceding_paragraphs.insert(0, prev_block)
            if preceding_paragraphs:
                preceding_text = "\n\n".join(preceding_paragraphs)
                preceding_str = f"**[Preceding Paragraphs]**\n{preceding_text}"
            

            # Grab paragraphs after the table (often table footnotes or continuation)
            succeeding_paragraphs = []
            for j in range(1, num_context_paragraphs + 1):
                if i + j < len(blocks):
                    next_block = blocks[i + j].strip()
                    if next_block:
                        succeeding_paragraphs.append(next_block)
            if succeeding_paragraphs:
                succeeding_text = "\n\n".join(succeeding_paragraphs)
                succeeding_str = f"**[Succeeding Paragraphs]**\n{succeeding_text}"


            surrounding_paragraphs.append((preceding_str, succeeding_str))

    return surrounding_paragraphs

In [ ]:
# Test
converter = DocumentConverter()
RAW_FILE = "./data/document.pdf"  # document per local path or URL
PARSED_FILE = "./data/clean_document.pdf"  # document per local path or URL

raw_result = converter.convert(RAW_FILE)
raw_markdown = raw_result.document.export_to_markdown()

redact_hyperlinked_text(RAW_FILE, PARSED_FILE)
parsed_result = converter.convert(PARSED_FILE)
parsed_markdown = parsed_result.document.export_to_markdown()


tables = get_tables(parsed_markdown)
surrounding_paragraphs = get_surrounding_paragraphs(raw_markdown, num_context_paragraphs=4)

with open('out.txt', 'w', encoding="utf-8") as output:
    for i in range(len(tables)-1):
        output.write(f"-------------- TABLE {i + 1} EXTRACTION --------------\n")
        output.write(f"{surrounding_paragraphs[i][0]}\n\n")
        output.write(f"{tables[i]}\n\n")
        output.write(surrounding_paragraphs[i][0])

print(f"Written to file 'out.txt'")


[INFO] 2026-06-23 15:16:46,872 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 15:16:46,880 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 15:16:46,881 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 15:16:47,060 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 15:16:47,063 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 15:16:47,064 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 15:16:47,129 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 15:16:47,140 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\mode

All hyperlinked elements redacted. Clean PDF saved.


[WARNING] 2026-06-23 15:17:10,629 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-06-23 15:17:10,958 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-06-23 15:17:11,388 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!


Written to file 'out.txt'
